# Component 3: Google Colab Standardized Baseline Model Training
**Project:** J26-DS-361 Intelligent MLOps Platform  
**Component:** Component 3 — Adaptive Intelligent Synthetic Data Repair Engine  
**Task:** Train and evaluate standardized baseline classification models (XGBoost, Random Forest, LightGBM) across real multi-dataset benchmarks with strictly leak-free preprocessing and joint model-preprocessor artifact serialization.  
**Execution Status:** NOT YET EXECUTED (Ready for Google Colab GPU Runtime)

---
### Research Panel Verification Checklist
- [x] **Real Tabular Benchmarks:** UCI Adult Census Income, Telco Customer Churn, Bank Marketing (Zero random demo data)
- [x] **Three Model Families:** XGBoost (`XGBClassifier`), Random Forest (`RandomForestClassifier`), LightGBM (`LGBMClassifier`)
- [x] **Duplicate-Safe Leakage Prevention:** Group-aware stratified splitting keeping identical feature records together (Zero duplicate feature vector leakage across train/val/test). Preprocessors fitted strictly on training data.
- [x] **Target & Schema Audit:** Verified binary targets, class imbalance weighting, and explicit exclusion of leakage-prone columns (`customerID`, `duration`)
- [x] **Joint Artifact Logging:** Saves `(model.joblib, preprocessor.joblib)` and metadata JSON records into Google Drive workspace
- [x] **Evaluation Visualizations:** Generates confusion matrices and multi-dataset model comparison charts


In [ ]:
# Step 1: Environment Setup & Hardware Acceleration Detection
import os
import sys
import platform
from pathlib import Path
import torch

print(f"Python Platform: {platform.platform()}")
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")

# GPU Detection with CPU Fallback
has_gpu = torch.cuda.is_available()
device = "cuda" if has_gpu else "cpu"
print(f"Compute Device: {device.upper()}")
if has_gpu:
    print(f"GPU Hardware: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
else:
    print("Notice: Running on CPU. Recommend switching to GPU in Runtime > Change runtime type for high-scale training.")

# Install required dependencies explicitly
!pip install --quiet xgboost scikit-learn lightgbm joblib pandas numpy matplotlib seaborn

# Setup repository access: if repo cloned in Colab, ensure root is in sys.path
if "google.colab" in sys.modules or os.path.exists("/content"):
    repo_dir = Path("/content/J26-DS-361-Intelligent-MLOps-Platform")
    if not repo_dir.exists():
        !git clone -b c3-mathusan https://github.com/sajivanK/J26-DS-361-Intelligent-MLOps-Platform.git
    if repo_dir.exists() and str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))
else:
    repo_dir = Path(".").resolve()
    if str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))

print("Environment setup and repository path resolution complete.")


In [ ]:
# Step 2: Google Drive Mounting & Workspace Configuration
from pathlib import Path

# Mount Drive with local fallback
try:
    from google.colab import drive
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive")
    DRIVE_WORKSPACE = Path("/content/drive/MyDrive/J26-DS-361-MLOps/component3")
except Exception as exc:
    print(f"Notice: Google Drive unavailable ({exc}). Using local workspace fallback.")
    DRIVE_WORKSPACE = Path("./colab_workspace").resolve()

# Establish directories
MODELS_DIR = DRIVE_WORKSPACE / "models"
PREPROCESSORS_DIR = DRIVE_WORKSPACE / "preprocessors"
METRICS_DIR = DRIVE_WORKSPACE / "metrics"
DATA_DIR = DRIVE_WORKSPACE / "data"
FIGURES_DIR = DRIVE_WORKSPACE / "figures"

for d in [MODELS_DIR, PREPROCESSORS_DIR, METRICS_DIR, DATA_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Workspace Directory: {DRIVE_WORKSPACE}")
print(f"Artifact Target: {MODELS_DIR}")


In [ ]:
# Step 3: Real Benchmark Dataset Ingestion & Schema Auditing
import pandas as pd
from component3.src.datasets.dataset_acquisition import DatasetAcquisitionManager
from component3.src.datasets.dataset_registry import get_dataset_registry

acq_manager = DatasetAcquisitionManager()
registry = get_dataset_registry()

# Target benchmarks to acquire and train
ACTIVE_DATASETS = ["uci_adult", "telco_churn", "bank_marketing"]
loaded_datasets = {}

print("=== INGESTING REAL BENCHMARK DATASETS ===")
for d_id in ACTIVE_DATASETS:
    meta = registry.get(d_id)
    raw_path, report = acq_manager.acquire_dataset(d_id)
    df = pd.read_csv(raw_path)
    loaded_datasets[d_id] = {
        "metadata": meta,
        "dataframe": df,
        "path": raw_path,
        "report": report,
    }

    print(f"\n[Dataset: {meta.display_name} ({d_id})]")
    print(f"  Source: {meta.source}")
    print(f"  Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"  Target Column: '{meta.target_column}' (Mapping: {meta.target_mapping})")
    print(f"  Target Distribution: {report.target_distribution}")
    print(f"  Numerical Features ({len(meta.numerical_features)}): {meta.numerical_features[:4]}...")
    print(f"  Categorical Features ({len(meta.categorical_features)}): {meta.categorical_features[:4]}...")
    if meta.leakage_prone_columns or meta.drop_columns:
        print(f"  Excluded Leakage/ID Columns: {meta.leakage_prone_columns + meta.drop_columns}")
    print(f"  Duplicate Rows: {report.duplicate_row_count} | Integrity Audit: {'PASS' if report.is_valid else 'WARNING'}")

print("\nAll real benchmark datasets ingested and verified successfully.")


In [ ]:
# Step 4: Leakage-Safe Stratified Splitting & Preprocessor Fitting
from component3.src.splitting.splitter import (
    DataSplitter,
    assert_no_split_leakage,
    assert_no_duplicate_feature_leakage,
)
from component3.src.preprocessing.tabular_preprocessor import get_preprocessor_for_dataset

prepared_splits = {}

print("=== PARTITIONING AND PREPROCESSING (FIT ON TRAIN ONLY) ===")
for d_id in ACTIVE_DATASETS:
    meta = loaded_datasets[d_id]["metadata"]
    df = loaded_datasets[d_id]["dataframe"]

    # 1. Reproducible, Duplicate-Safe Partitioning: 70% Train, 15% Validation, 15% Test
    # Identical feature records are kept strictly in the same partition to prevent memorization leakage
    splits, manifest = DataSplitter.split(
        df=df,
        metadata=meta,
        train_ratio=0.70,
        val_ratio=0.15,
        test_ratio=0.15,
        duplicate_safe=True,
        seed=42,
    )

    # 2. Strict Scientific Leakage Verification Checks
    assert_no_split_leakage(manifest)
    leakage_audit = assert_no_duplicate_feature_leakage(
        splits,
        feature_columns=meta.feature_columns,
        missing_representations=meta.missing_representations,
    )

    # 3. Fit Preprocessor EXCLUSIVELY on Training Split
    preprocessor = get_preprocessor_for_dataset(d_id)
    train_bundle = preprocessor.fit_transform(splits.train)

    # 4. Transform Validation and Test Splits without parameter updates
    val_bundle = preprocessor.transform(splits.val)
    test_bundle = preprocessor.transform(splits.test)

    # 5. Persist Split Manifest to Google Drive Workspace
    manifest_path = METRICS_DIR / f"{d_id}_split_manifest.json"
    manifest.save(manifest_path)

    prepared_splits[d_id] = {
        "preprocessor": preprocessor,
        "splits": splits,
        "manifest": manifest,
        "train_bundle": train_bundle,
        "val_bundle": val_bundle,
        "test_bundle": test_bundle,
    }

    print(f"\n[{d_id.upper()} Partitions (Duplicate-Safe Group Splitting)]")
    print(f"  Dataset Fingerprint: {manifest.dataset_fingerprint}")
    print(f"  Rows in Duplicate Feature Groups: {manifest.rows_in_duplicate_groups:,} ({manifest.duplicate_group_count:,} unique groups)")
    print(f"  Cross-Partition Feature Overlap: {leakage_audit['cross_partition_overlap']} (VERIFIED 0% LEAKAGE)")
    print(f"  Train Split: {len(splits.train):,} rows ({len(splits.train)/len(df):.1%}) -> Matrix X: {train_bundle.X.shape}")
    print(f"  Val Split:   {len(splits.val):,} rows ({len(splits.val)/len(df):.1%}) -> Matrix X: {val_bundle.X.shape}")
    print(f"  Test Split:  {len(splits.test):,} rows ({len(splits.test)/len(df):.1%}) -> Matrix X: {test_bundle.X.shape}")
    print(f"  Feature Dim: {train_bundle.X.shape[1]} engineered features (Scaling + Encoding applied)")
    print(f"  Class Proportions: Train={manifest.class_proportions.get('train')} | Val={manifest.class_proportions.get('val')} | Test={manifest.class_proportions.get('test')}")

print("\nZero-leakage partitioning and preprocessing complete.")


In [ ]:
# Step 5: Multi-Model Baseline Training (XGBoost, Random Forest, LightGBM)
import json
import joblib
from component3.src.baselines.baseline_trainer import BaselineModelTrainer, ModelFamily

MODEL_FAMILIES = [
    ModelFamily.XGBOOST,
    ModelFamily.RANDOM_FOREST,
    ModelFamily.LIGHTGBM,
]

training_results = []

print("=== EXECUTING MULTI-MODEL BASELINE TRAINING ===")
for d_id in ACTIVE_DATASETS:
    prep_data = prepared_splits[d_id]
    preprocessor = prep_data["preprocessor"]
    meta = loaded_datasets[d_id]["metadata"]

    tb = prep_data["train_bundle"]
    vb = prep_data["val_bundle"]
    test_b = prep_data["test_bundle"]

    for fam in MODEL_FAMILIES:
        exp_id = f"{d_id}_{fam.value}_baseline_seed42"
        print(f"\nTraining [{d_id}] with [{fam.value.upper()}]...")

        # Joint artifact filenames
        model_file = MODELS_DIR / f"{exp_id}_model.joblib"
        prep_file = PREPROCESSORS_DIR / f"{exp_id}_preprocessor.joblib"
        metrics_file = METRICS_DIR / f"{exp_id}_metrics.json"

        # Train and evaluate using standardized trainer
        model, record = BaselineModelTrainer.train_and_evaluate(
            dataset_id=d_id,
            family=fam,
            X_train=tb.X,
            y_train=tb.y,
            X_val=vb.X,
            y_val=vb.y,
            X_test=test_b.X,
            y_test=test_b.y,
            seed=42,
            class_weights=meta.class_distribution,
            model_save_path=model_file,
        )

        # Crucial Requirement: Model and Preprocessor saved TOGETHER
        joblib.dump(preprocessor, prep_file)

        # Save metadata and metrics
        record.save(metrics_file)

        print(f"  Val Metrics  -> F1: {record.val_metrics.f1:.4f} | Acc: {record.val_metrics.accuracy:.4f} | AUC: {record.val_metrics.roc_auc:.4f}")
        print(f"  Test Metrics -> F1: {record.test_metrics.f1:.4f} | Acc: {record.test_metrics.accuracy:.4f} | AUC: {record.test_metrics.roc_auc:.4f}")
        print(f"  Artifacts Saved: {model_file.name} + {prep_file.name}")

        training_results.append({
            "experiment_id": exp_id,
            "dataset_id": d_id,
            "model_family": fam.value,
            "record": record,
            "model_file": str(model_file),
            "prep_file": str(prep_file),
            "metrics_file": str(metrics_file),
        })

print(f"\nBaseline training completed: {len(training_results)} total model artifacts saved jointly.")


In [ ]:
# Step 6: Real Evaluation Plots & Model Performance Comparison
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Comparative Performance Table
summary_rows = []
for res in training_results:
    rec = res["record"]
    summary_rows.append({
        "Dataset": res["dataset_id"],
        "Model": res["model_family"].upper(),
        "Val F1": rec.val_metrics.f1,
        "Test F1": rec.test_metrics.f1,
        "Test Accuracy": rec.test_metrics.accuracy,
        "Test ROC-AUC": rec.test_metrics.roc_auc,
    })

summary_df = pd.DataFrame(summary_rows)
print("=== BASELINE PERFORMANCE SUMMARY ===")
print(summary_df.to_string(index=False))

# 2. Multi-Dataset Model Comparison Bar Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# Test F1 Plot
sns.barplot(
    data=summary_df,
    x="Dataset",
    y="Test F1",
    hue="Model",
    palette=["#2b5c8f", "#e07a5f", "#81b29a"],
    ax=axes[0]
)
axes[0].set_title("Baseline Comparison: Test F1 Score", fontsize=12, fontweight="bold")
axes[0].set_ylim(0.0, 1.0)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

# Test ROC-AUC Plot
sns.barplot(
    data=summary_df,
    x="Dataset",
    y="Test ROC-AUC",
    hue="Model",
    palette=["#2b5c8f", "#e07a5f", "#81b29a"],
    ax=axes[1]
)
axes[1].set_title("Baseline Comparison: Test ROC-AUC", fontsize=12, fontweight="bold")
axes[1].set_ylim(0.5, 1.0)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
comp_fig_path = FIGURES_DIR / "baseline_model_comparison.png"
fig.savefig(comp_fig_path)
plt.show()
print(f"Comparison figure saved to: {comp_fig_path}")

# 3. Confusion Matrix Visualization for XGBoost Baselines
fig_cm, axes_cm = plt.subplots(1, len(ACTIVE_DATASETS), figsize=(15, 4), dpi=150)
for idx, d_id in enumerate(ACTIVE_DATASETS):
    xgb_rec = [r["record"] for r in training_results if r["dataset_id"] == d_id and r["model_family"] == "xgboost"][0]
    cm = np.array(xgb_rec.test_metrics.confusion_matrix)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes_cm[idx])
    axes_cm[idx].set_title(f"{d_id.upper()} (XGBoost)", fontsize=11, fontweight="bold")
    axes_cm[idx].set_xlabel("Predicted Label")
    axes_cm[idx].set_ylabel("True Label")

plt.tight_layout()
cm_fig_path = FIGURES_DIR / "baseline_confusion_matrices.png"
fig_cm.savefig(cm_fig_path)
plt.show()
print(f"Confusion matrices saved to: {cm_fig_path}")


In [ ]:
# Step 7: Verification Checks & Save/Reload Consistency
from component3.src.baselines.model_adapter import BaselineModelAdapter

print("=== VERIFYING MODEL & PREPROCESSOR SAVE/RELOAD CONSISTENCY ===")
all_passed = True

for res in training_results:
    d_id = res["dataset_id"]
    fam = res["model_family"]
    original_rec = res["record"]

    # Reload model and preprocessor using BaselineModelAdapter
    adapter = BaselineModelAdapter.load(
        model_path=res["model_file"],
        model_family=fam,
        preprocessor_path=res["prep_file"],
    )

    # Re-evaluate on untouched raw test split
    test_raw_df = prepared_splits[d_id]["splits"].test
    target_col = loaded_datasets[d_id]["metadata"].target_column
    reloaded_metrics = adapter.evaluate(test_raw_df, test_raw_df[target_col])

    # Assert metric consistency between trainer and reloaded adapter
    f1_diff = abs(reloaded_metrics.f1 - original_rec.test_metrics.f1)
    acc_diff = abs(reloaded_metrics.accuracy - original_rec.test_metrics.accuracy)

    status = "PASS" if (f1_diff < 1e-4 and acc_diff < 1e-4) else "FAIL"
    if status == "FAIL":
        all_passed = False

    print(f"[{status}] {d_id} ({fam}): Original F1={original_rec.test_metrics.f1:.4f}, Reloaded F1={reloaded_metrics.f1:.4f} (diff={f1_diff:.6f})")

print(f"\nConsistency Verification: {'ALL CHECKS PASSED' if all_passed else 'FAILURES DETECTED'}")
print("Baseline models and fitted preprocessors are certified ready for controlled drift experiments.")
